In [ ]:
from dotenv import load_dotenv
import os

load_dotenv()

In [ ]:
from chains.extract_chain import extract_chain
from chains.match_chain import match_chain
from chains.score_chain import score_chain
from chains.explain_chain import explain_chain

In [ ]:
def load_file(path):
    with open(path, "r") as f:
        return f.read()

jd = load_file("data/job_description.txt")

In [ ]:
import json
import re

def run_pipeline(resume_path):
    resume = load_file(resume_path)

    # Step 1: Extract
    resume_data = extract_chain.invoke(
        {"resume": resume},
        config={"run_name": "extract"}
    )

    # Step 2: Match
    match_data = match_chain.invoke(
        {
            "resume_data": resume_data,
            "job_description": jd
        },
        config={"run_name": "match"}
    )

    # Step 3: Score
    score_raw = score_chain.invoke(
        {"match_data": match_data},
        config={"run_name": "score"}
    )

    print("RAW SCORE OUTPUT:", score_raw)

    # Clean markdown ```json ```
    cleaned = re.sub(r"```json|```", "", score_raw).strip()

    try:
        score = json.loads(cleaned)["score"]
    except:
        match = re.search(r"\d{1,3}", cleaned)
        score = int(match.group()) if match else 0

    # Step 4: Explain
    explanation = explain_chain.invoke(
        {
            "score": score,
            "match_data": match_data
        },
        config={"run_name": "explain"}
    )

    print("\n===== RESULT =====")
    print("Score:", score)
    print("Explanation:", explanation)

    return score, explanation

from langchain_core.runnables import RunnableLambda

pipeline = RunnableLambda(
    lambda resume_path: run_pipeline(resume_path)
).with_config({"run_name": "resume_pipeline"})

In [ ]:
print("Strong Candidate")
pipeline.invoke("data/strong_resume.txt")